# TP4 - Distribution-Aligned Sequence Distillation (DASD)
## Theme : Pokemon

Ce notebook implemente la methode DASD pour distiller les capacites de raisonnement d'un LLM enseignant (API Infomaniak) vers un modele etudiant compact (Qwen3-4B) via Llama-Factory.

**Pipeline :**
1. Generation d'un dataset Pokemon via l'API enseignant (2 stages de temperature)
2. Divergence-Aware Sampling (DAS) pour filtrer les donnees
3. Entrainement LoRA en 2 stages avec Llama-Factory
4. Evaluation du modele distille

---
## Phase 1 : Setup

In [ ]:
# Cellule 1 : Verification GPU
!nvidia-smi

In [ ]:
# Cellule 2 : Installation de Llama-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,bitsandbytes]" -q
!llamafactory-cli version

In [ ]:
# Cellule 3 : Imports et configuration
import json
import os
import time
import gc
import re
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
import nltk
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Configuration API Infomaniak
from google.colab import userdata

API_KEY = userdata.get('INFOMANIAK_API_KEY')
BASE_URL = "https://api.infomaniak.com/2/ai/48/openai/v1"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Repertoire de travail
WORK_DIR = Path("/content/LLaMA-Factory")
DATA_DIR = WORK_DIR / "data"

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cellule 4 : Listing des modeles disponibles sur l'API Infomaniak
models = client.models.list()
print("Modeles disponibles :")
for m in models.data:
    print(f"  - {m.id}")

# Modele enseignant : gpt-oss-120b (meme modele que le dataset de reference Alibaba DASD)
TEACHER_MODEL = "openai/gpt-oss-120b"
print(f"\nModele enseignant selectionne : {TEACHER_MODEL}")

---
## Phase 2 : Etude du dataset de reference DASD

In [ ]:
# Cellule 5 : Chargement du dataset de reference Alibaba DASD
from datasets import load_dataset

reference_dataset = load_dataset(
    "Alibaba-Apsara/Superior-Reasoning-SFT-gpt-oss-120b",
    "stage1",
    split="train"
)

print(f"Nombre total d'exemples : {len(reference_dataset)}")
print(f"Colonnes : {reference_dataset.column_names}")
print("\n" + "="*80)

# Affichage de 5 exemples
for i in range(min(5, len(reference_dataset))):
    example = reference_dataset[i]
    print(f"\n--- Exemple {i+1} ---")
    for key, value in example.items():
        preview = str(value)[:300]
        print(f"{key}: {preview}..." if len(str(value)) > 300 else f"{key}: {preview}")

In [ ]:
# Cellule 6 : Analyse du format <think>...</think>
print("Analyse du format de raisonnement dans le dataset de reference\n")

think_count = 0
reasoning_count = 0
total_checked = min(100, len(reference_dataset))

for i in range(total_checked):
    example = reference_dataset[i]
    text = str(example)
    if "<think>" in text:
        think_count += 1
    if "<reasoning>" in text:
        reasoning_count += 1

print(f"Sur {total_checked} exemples :")
print(f"  - Contenant <think>...</think> : {think_count}")
print(f"  - Contenant <reasoning>...</reasoning> : {reasoning_count}")

# Afficher un exemple complet de raisonnement
print("\n" + "="*80)
print("Exemple complet de reponse structuree :")
example = reference_dataset[0]
for key in example:
    val = str(example[key])
    if len(val) > 500:
        print(f"\n[{key}] (premiers 500 chars) :\n{val[:500]}...")
    else:
        print(f"\n[{key}] :\n{val}")

---
## Phase 3 : Generation du dataset Pokemon

In [ ]:
# Cellule 7 : Definition des 28 questions Pokemon en francais

POKEMON_QUESTIONS = {
    "efficacite_types": [
        "Pourquoi un Pokemon de type Sol est-il particulierement efficace contre un Pokemon Electrik/Acier comme Magnezone ? Explique les doubles faiblesses et immunites en jeu.",
        "Un Dracolosse (Dragon/Vol) affronte un Artikodin (Glace/Vol). Analyse les interactions de types offensives et defensives pour chaque cote.",
        "Explique le concept de STAB (Same Type Attack Bonus) et calcule les degats relatifs d'un Lance-Flammes utilise par Dracaufeu (Feu/Vol) versus par Mewtwo (Psy).",
        "Un Carchacrok (Dragon/Sol) utilise Seisme contre un Noctali (Tenebres). Puis il utilise Draco-Griffe. Compare l'efficacite de ces deux attaques en tenant compte du STAB.",
        "Pourquoi le type Fee a-t-il ete introduit en Generation 6 ? Analyse son impact sur l'equilibre competitif, notamment contre les types Dragon, Combat et Tenebres.",
        "Un Pokemon de type Normal peut-il toucher un Pokemon Spectre ? Explique les immunites de type et les capacites qui peuvent contourner cette regle."
    ],
    "chaines_evolution": [
        "Decris toutes les evolutions possibles d'Evoli et les conditions requises pour chacune. Quelle est la plus utile en competitif et pourquoi ?",
        "Compare les chaines d'evolution de Reptincel et Carchacrok. A quel stade chaque Pokemon apprend-il ses capacites cles ?",
        "Explique le mecanisme de la Mega-Evolution avec l'exemple de Dracaufeu. Pourquoi a-t-il deux Mega-Evolutions (X et Y) et quelles sont les differences strategiques ?",
        "Comment fonctionne l'evolution par echange ? Donne 3 exemples de Pokemon qui evoluent par echange et explique pourquoi ce mecanisme existe dans le jeu.",
        "Pourquoi certains Pokemon comme Magikarp sont-ils volontairement faibles avant evolution ? Analyse la courbe de progression de Magikarp vers Leviator en termes de stats."
    ],
    "comparaison_stats": [
        "Compare les stats de base (BST) de Dracaufeu, Tortank et Florizarre. Lequel est le plus equilibre et pourquoi ?",
        "Qu'est-ce qu'un speed tier en competitif Pokemon ? Compare les vitesses de Electrode, Ninjask et Deoxys-Speed et explique pourquoi la vitesse est si importante.",
        "Analyse les stats de Mackogneur : pourquoi est-il considere comme un 'glass cannon' malgre sa haute Attaque ? Quelles sont ses faiblesses statistiques ?",
        "Compare Leveinard et Leuphorie en tant que murs speciaux. Quelles stats les rendent si efficaces dans ce role et quelles sont leurs faiblesses communes ?",
        "Pourquoi Archeodong est-il considere comme un bon Pokemon defensif ? Analyse son typing et ses stats defensives.",
        "Compare les BST des starters finaux de chaque generation (Gen 1 a 4). Y a-t-il une tendance dans la distribution des stats ?"
    ],
    "team_building": [
        "Construis une equipe de 6 Pokemon pour le format OU (Overused) qui couvre toutes les faiblesses de type. Explique la synergie entre chaque membre.",
        "Qu'est-ce qu'un core defensif FEE (Feu/Eau/Electrik) et pourquoi est-il si populaire en competitif ? Donne un exemple concret avec des Pokemon.",
        "Comment contrer une strategie basee sur la Danse du Dragon ? Propose 3 contre-mesures differentes avec des Pokemon specifiques.",
        "Explique le role de chaque position dans une equipe competitive : lead, sweeper, wall, pivot, revenge killer et support. Donne un Pokemon typique pour chaque role.",
        "Construis une equipe 'balance' autour de Carchacrok comme sweeper principal. Quels Pokemon complementent ses faiblesses (Glace 4x, Dragon, Fee) ?",
        "Pourquoi les hazards d'entree (Picots, Rochers Furtifs) sont-ils si importants en competitif ? Quel impact ont les Rochers Furtifs sur les Pokemon de type Vol/Feu ?"
    ],
    "calcul_degats": [
        "Calcule les degats approximatifs d'un Seisme (puissance 100) utilise par un Carchacrok (Attaque base 130) niveau 100 contre un Electhor (Defense base 85). Montre chaque etape du calcul.",
        "Explique la formule de calcul des degats en Pokemon (Generation 4+). Quels sont les multiplicateurs qui s'appliquent et dans quel ordre ?",
        "Un Dracaufeu Mega-Evolue Y utilise Lance-Soleil sous Secheresse. Calcule le multiplicateur total en tenant compte de STAB, Secheresse, et l'efficacite contre un Pokemon Eau/Sol.",
        "Compare les degats d'un Hydrocanon (puissance 150, precision 80%) versus Surf (puissance 90, precision 100%) sur 5 tours. Quelle attaque est la plus rentable et pourquoi ?",
        "Explique comment les natures et les EVs affectent les stats finales d'un Pokemon. Calcule la stat d'Attaque d'un Carchacrok niveau 100 avec nature Jovial, 252 EVs et 31 IVs en Attaque."
    ]
}

# Liste plate de toutes les questions
all_questions = []
for category, questions in POKEMON_QUESTIONS.items():
    for q in questions:
        all_questions.append({"category": category, "question": q})

print(f"Total de questions : {len(all_questions)}")
for cat, qs in POKEMON_QUESTIONS.items():
    print(f"  {cat}: {len(qs)} questions")

In [ ]:
# Cellule 8 : Fonction de generation des reponses enseignant

SYSTEM_PROMPT = (
    "Tu es un expert Pokemon competitif. "
    "Pour chaque question, raisonne etape par etape a l'interieur de balises <reasoning>...</reasoning> "
    "avant de donner ta reponse finale. "
    "Sois precis, utilise les vrais noms francais des Pokemon, et structure clairement ton raisonnement "
    "avec des etapes numerotees."
)

def generate_teacher_response(question, temperature=0.3, max_retries=3):
    """
    Appelle l'API enseignant pour generer une reponse avec logprobs.
    Inclut retry avec backoff exponentiel et filtre qualite.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=TEACHER_MODEL,
                messages=messages,
                temperature=temperature,
                max_tokens=4096,
                logprobs=True,
                top_logprobs=1
            )

            content = response.choices[0].message.content
            logprobs_data = response.choices[0].logprobs

            # Filtre qualite : longueur minimale et presence de raisonnement
            if len(content) < 100:
                print(f"  [Retry {attempt+1}] Reponse trop courte ({len(content)} chars)")
                time.sleep(2 ** attempt)
                continue

            # Serialisation des logprobs
            serialized_logprobs = []
            if logprobs_data and logprobs_data.content:
                for token_data in logprobs_data.content:
                    serialized_logprobs.append({
                        "token": token_data.token,
                        "logprob": token_data.logprob
                    })

            return {
                "content": content,
                "logprobs": serialized_logprobs,
                "has_reasoning": "<reasoning>" in content and "</reasoning>" in content,
                "temperature": temperature
            }

        except Exception as e:
            wait = 2 ** (attempt + 1)
            print(f"  [Retry {attempt+1}] Erreur: {e} - attente {wait}s")
            time.sleep(wait)

    print(f"  ECHEC apres {max_retries} tentatives")
    return None

# Test rapide
test_response = generate_teacher_response("Quel est le type de Dracaufeu ?", temperature=0.3)
if test_response:
    print(f"Test OK - Longueur: {len(test_response['content'])} chars")
    print(f"Raisonnement structure: {test_response['has_reasoning']}")
    print(f"Nombre de tokens avec logprobs: {len(test_response['logprobs'])}")
    print(f"\nDebut de la reponse:\n{test_response['content'][:300]}...")

In [ ]:
# Cellule 9 : Generation Stage 1 (temperature basse = 0.3)

print("=" * 60)
print("STAGE 1 : Generation a basse temperature (tau=0.3)")
print("=" * 60)

stage1_raw = []

for i, item in enumerate(all_questions):
    print(f"\n[{i+1}/{len(all_questions)}] {item['category']} - {item['question'][:60]}...")
    result = generate_teacher_response(item["question"], temperature=0.3)

    if result:
        stage1_raw.append({
            "category": item["category"],
            "question": item["question"],
            "response": result["content"],
            "logprobs": result["logprobs"],
            "has_reasoning": result["has_reasoning"],
            "temperature": result["temperature"]
        })
        print(f"  OK - {len(result['content'])} chars, reasoning={result['has_reasoning']}")
    else:
        print(f"  ECHEC")

    time.sleep(1)  # Rate limiting

print(f"\nStage 1 : {len(stage1_raw)}/{len(all_questions)} reponses generees")
print(f"Avec raisonnement structure : {sum(1 for r in stage1_raw if r['has_reasoning'])}")

# Sauvegarde
with open(WORK_DIR / "stage1_raw.json", "w", encoding="utf-8") as f:
    json.dump(stage1_raw, f, ensure_ascii=False, indent=2)
print(f"Sauvegarde dans stage1_raw.json")

In [ ]:
# Cellule 10 : Generation Stage 2 (temperature haute = 0.9)

print("=" * 60)
print("STAGE 2 : Generation a haute temperature (tau=0.9)")
print("=" * 60)

stage2_raw = []

for i, item in enumerate(all_questions):
    print(f"\n[{i+1}/{len(all_questions)}] {item['category']} - {item['question'][:60]}...")
    result = generate_teacher_response(item["question"], temperature=0.9)

    if result:
        stage2_raw.append({
            "category": item["category"],
            "question": item["question"],
            "response": result["content"],
            "logprobs": result["logprobs"],
            "has_reasoning": result["has_reasoning"],
            "temperature": result["temperature"]
        })
        print(f"  OK - {len(result['content'])} chars, reasoning={result['has_reasoning']}")
    else:
        print(f"  ECHEC")

    time.sleep(1)  # Rate limiting

print(f"\nStage 2 : {len(stage2_raw)}/{len(all_questions)} reponses generees")
print(f"Avec raisonnement structure : {sum(1 for r in stage2_raw if r['has_reasoning'])}")

# Sauvegarde
with open(WORK_DIR / "stage2_raw.json", "w", encoding="utf-8") as f:
    json.dump(stage2_raw, f, ensure_ascii=False, indent=2)
print(f"Sauvegarde dans stage2_raw.json")

In [ ]:
# Cellule 11 : Statistiques de generation

def print_stats(data, stage_name):
    print(f"\n{'='*40}")
    print(f"Statistiques {stage_name}")
    print(f"{'='*40}")
    print(f"Nombre d'exemples : {len(data)}")

    lengths = [len(d['response']) for d in data]
    reasoning_count = sum(1 for d in data if d['has_reasoning'])
    logprob_counts = [len(d['logprobs']) for d in data]

    print(f"Avec <reasoning> : {reasoning_count}/{len(data)} ({100*reasoning_count/len(data):.0f}%)")
    print(f"Longueur reponse - min: {min(lengths)}, max: {max(lengths)}, moy: {np.mean(lengths):.0f}")
    print(f"Tokens logprobs  - min: {min(logprob_counts)}, max: {max(logprob_counts)}, moy: {np.mean(logprob_counts):.0f}")

    # Par categorie
    categories = {}
    for d in data:
        cat = d['category']
        if cat not in categories:
            categories[cat] = []
        categories[cat].append(len(d['response']))

    print(f"\nPar categorie :")
    for cat, lens in categories.items():
        print(f"  {cat}: {len(lens)} exemples, longueur moyenne {np.mean(lens):.0f} chars")

print_stats(stage1_raw, "Stage 1 (tau=0.3)")
print_stats(stage2_raw, "Stage 2 (tau=0.9)")

In [ ]:
# Cellule 12 : Apercu d'une reponse generee

print("Exemple de reponse Stage 1 :\n")
example = stage1_raw[0]
print(f"Categorie : {example['category']}")
print(f"Question : {example['question']}")
print(f"\nReponse ({len(example['response'])} chars) :")
print(example['response'][:1000])
if len(example['response']) > 1000:
    print(f"\n... ({len(example['response']) - 1000} chars de plus)")

print(f"\nLogprobs (5 premiers tokens) :")
for lp in example['logprobs'][:5]:
    print(f"  token='{lp['token']}', logprob={lp['logprob']:.4f}")

---
## Phase 4 : Implementation du Divergence-Aware Sampling (DAS)

In [ ]:
# Cellule 13 : Chargement du modele etudiant Qwen3-4B en 4-bit

STUDENT_MODEL_ID = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Chargement du modele etudiant : {STUDENT_MODEL_ID}...")
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
student_model.eval()

print(f"Modele charge sur {student_model.device}")
print(f"Memoire GPU utilisee : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cellule 14 : Fonction de calcul des scores DAS
# Adaptee de simple_dasd.py

def calculate_das_scores(question, teacher_text, teacher_logprobs_list):
    """
    Calcule les scores DAS phrase par phrase.
    Adapte de simple_dasd.py : calculate_sentence_scores()

    Args:
        question: la question posee
        teacher_text: le texte de la reponse du teacher
        teacher_logprobs_list: liste de dicts {"token": str, "logprob": float}

    Returns:
        liste de dicts par phrase avec p_teacher, p_student, divergence, classification
    """
    # A. Preparation du prompt etudiant
    messages = [{"role": "user", "content": question}]
    prompt_str = student_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    full_input_str = prompt_str + teacher_text

    # B. Forward pass etudiant
    inputs = student_tokenizer(full_input_str, return_tensors="pt").to(student_model.device)
    prompt_tokens_len = len(student_tokenizer(prompt_str, add_special_tokens=False)["input_ids"])

    with torch.no_grad():
        outputs = student_model(**inputs)
        logits = outputs.logits

    # Shift logits : logits[i] predit input[i+1]
    shift_logits = logits[0, :-1, :]
    shift_labels = inputs["input_ids"][0, 1:]

    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_losses = loss_fct(shift_logits, shift_labels)
    token_logprobs_student = -token_losses.cpu().numpy()

    # Partie reponse uniquement
    response_logprobs_student = token_logprobs_student[prompt_tokens_len - 1:]
    response_token_ids = shift_labels[prompt_tokens_len - 1:].cpu().numpy()

    # C. Decoupage en phrases et alignement
    sentences = nltk.tokenize.sent_tokenize(teacher_text)
    results = []

    openai_cursor = 0
    qwen_cursor = 0

    for sent in sentences:
        # Score Teacher
        current_sent_accum = ""
        sent_teacher_logprobs = []

        while openai_cursor < len(teacher_logprobs_list):
            t_data = teacher_logprobs_list[openai_cursor]
            sent_teacher_logprobs.append(t_data["logprob"])
            current_sent_accum += t_data["token"]
            openai_cursor += 1
            if len(current_sent_accum) >= len(sent):
                break

        p_teacher = float(np.exp(np.mean(sent_teacher_logprobs))) if sent_teacher_logprobs else 0.0

        # Score Student
        current_sent_accum_qwen = ""
        sent_student_logprobs = []

        while qwen_cursor < len(response_token_ids):
            tid = response_token_ids[qwen_cursor]
            token_str = student_tokenizer.decode([tid])
            sent_student_logprobs.append(response_logprobs_student[qwen_cursor])
            current_sent_accum_qwen += token_str
            qwen_cursor += 1
            if len(current_sent_accum_qwen) >= len(sent):
                break

        p_student = float(np.exp(np.mean(sent_student_logprobs))) if sent_student_logprobs else 0.0

        divergence = p_teacher - p_student

        # Classification
        if p_teacher > 0.6 and divergence > 0.2:
            classification = "Teacher Sentence"
        elif divergence < -0.1:
            classification = "Student Sentence"
        else:
            classification = "Shared"

        results.append({
            "sentence": sent,
            "p_teacher": p_teacher,
            "p_student": p_student,
            "divergence": divergence,
            "classification": classification
        })

    return results

print("Fonction calculate_das_scores() definie.")

In [ ]:
# Cellule 15 : Calcul DAS pour le Stage 1

print("Calcul des scores DAS pour Stage 1...\n")

stage1_das_results = []

for i, item in enumerate(stage1_raw):
    print(f"[{i+1}/{len(stage1_raw)}] {item['question'][:50]}...", end=" ")
    try:
        scores = calculate_das_scores(
            item["question"],
            item["response"],
            item["logprobs"]
        )

        # Score agrege : moyenne des divergences
        avg_divergence = np.mean([s["divergence"] for s in scores])
        teacher_count = sum(1 for s in scores if s["classification"] == "Teacher Sentence")
        student_count = sum(1 for s in scores if s["classification"] == "Student Sentence")

        stage1_das_results.append({
            "index": i,
            "question": item["question"],
            "response": item["response"],
            "category": item["category"],
            "sentence_scores": scores,
            "avg_divergence": float(avg_divergence),
            "teacher_sentences": teacher_count,
            "student_sentences": student_count,
            "total_sentences": len(scores)
        })

        print(f"OK - div={avg_divergence:.3f}, teacher={teacher_count}, student={student_count}")

    except Exception as e:
        print(f"ERREUR: {e}")

print(f"\nDAS Stage 1 : {len(stage1_das_results)} exemples traites")

In [ ]:
# Cellule 16 : Calcul DAS pour le Stage 2

print("Calcul des scores DAS pour Stage 2...\n")

stage2_das_results = []

for i, item in enumerate(stage2_raw):
    print(f"[{i+1}/{len(stage2_raw)}] {item['question'][:50]}...", end=" ")
    try:
        scores = calculate_das_scores(
            item["question"],
            item["response"],
            item["logprobs"]
        )

        avg_divergence = np.mean([s["divergence"] for s in scores])
        teacher_count = sum(1 for s in scores if s["classification"] == "Teacher Sentence")
        student_count = sum(1 for s in scores if s["classification"] == "Student Sentence")

        stage2_das_results.append({
            "index": i,
            "question": item["question"],
            "response": item["response"],
            "category": item["category"],
            "sentence_scores": scores,
            "avg_divergence": float(avg_divergence),
            "teacher_sentences": teacher_count,
            "student_sentences": student_count,
            "total_sentences": len(scores)
        })

        print(f"OK - div={avg_divergence:.3f}, teacher={teacher_count}, student={student_count}")

    except Exception as e:
        print(f"ERREUR: {e}")

print(f"\nDAS Stage 2 : {len(stage2_das_results)} exemples traites")

In [ ]:
# Cellule 17 : Visualisations DAS

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogramme des divergences moyennes
ax = axes[0, 0]
divs1 = [r["avg_divergence"] for r in stage1_das_results]
divs2 = [r["avg_divergence"] for r in stage2_das_results]
ax.hist(divs1, bins=15, alpha=0.6, label="Stage 1 (tau=0.3)", color="steelblue")
ax.hist(divs2, bins=15, alpha=0.6, label="Stage 2 (tau=0.9)", color="coral")
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel("Divergence moyenne")
ax.set_ylabel("Nombre d'exemples")
ax.set_title("Distribution des divergences DAS")
ax.legend()

# 2. Scatter P_teacher vs P_student (Stage 1)
ax = axes[0, 1]
all_pt1 = [s["p_teacher"] for r in stage1_das_results for s in r["sentence_scores"]]
all_ps1 = [s["p_student"] for r in stage1_das_results for s in r["sentence_scores"]]
ax.scatter(all_pt1, all_ps1, alpha=0.3, s=10, color="steelblue")
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5)
ax.set_xlabel("P_teacher")
ax.set_ylabel("P_student")
ax.set_title("Stage 1 : P_teacher vs P_student (par phrase)")

# 3. Scatter P_teacher vs P_student (Stage 2)
ax = axes[1, 0]
all_pt2 = [s["p_teacher"] for r in stage2_das_results for s in r["sentence_scores"]]
all_ps2 = [s["p_student"] for r in stage2_das_results for s in r["sentence_scores"]]
ax.scatter(all_pt2, all_ps2, alpha=0.3, s=10, color="coral")
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5)
ax.set_xlabel("P_teacher")
ax.set_ylabel("P_student")
ax.set_title("Stage 2 : P_teacher vs P_student (par phrase)")

# 4. Classification des phrases
ax = axes[1, 1]
for stage_name, results, color in [("Stage 1", stage1_das_results, "steelblue"),
                                     ("Stage 2", stage2_das_results, "coral")]:
    classifications = [s["classification"] for r in results for s in r["sentence_scores"]]
    labels = ["Teacher", "Shared", "Student"]
    counts = [classifications.count(f"{l} Sentence") if l != "Shared" else classifications.count("Shared") for l in labels]
    x = np.arange(len(labels))
    offset = -0.2 if stage_name == "Stage 1" else 0.2
    ax.bar(x + offset, counts, 0.35, label=stage_name, color=color, alpha=0.7)

ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Nombre de phrases")
ax.set_title("Classification des phrases par type")
ax.legend()

plt.tight_layout()
plt.savefig(WORK_DIR / "das_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("Visualisation sauvegardee dans das_analysis.png")

In [ ]:
# Cellule 18 : Filtrage DAS et conversion au format Alpaca

def filter_and_convert(das_results, stage_name):
    """
    Filtre les exemples avec score DAS >= 0 (divergence positive)
    et convertit au format Alpaca pour Llama-Factory.
    """
    filtered = [r for r in das_results if r["avg_divergence"] >= 0]

    alpaca_data = []
    for r in filtered:
        alpaca_data.append({
            "instruction": r["question"],
            "input": "",
            "output": r["response"]
        })

    print(f"{stage_name}: {len(filtered)}/{len(das_results)} exemples retenus apres filtrage DAS")
    print(f"  Rejetes (divergence < 0): {len(das_results) - len(filtered)}")

    return alpaca_data

# Filtrage et conversion
pokemon_stage1 = filter_and_convert(stage1_das_results, "Stage 1")
pokemon_stage2 = filter_and_convert(stage2_das_results, "Stage 2")

# Sauvegarde au format Alpaca dans le repertoire data de Llama-Factory
with open(DATA_DIR / "pokemon_stage1.json", "w", encoding="utf-8") as f:
    json.dump(pokemon_stage1, f, ensure_ascii=False, indent=2)

with open(DATA_DIR / "pokemon_stage2.json", "w", encoding="utf-8") as f:
    json.dump(pokemon_stage2, f, ensure_ascii=False, indent=2)

print(f"\nFichiers sauvegardes :")
print(f"  data/pokemon_stage1.json ({len(pokemon_stage1)} exemples)")
print(f"  data/pokemon_stage2.json ({len(pokemon_stage2)} exemples)")

# Apercu
print(f"\nApercu d'un exemple Stage 1 :")
print(json.dumps(pokemon_stage1[0], ensure_ascii=False, indent=2)[:500])

---
## Phase 5 : Configuration de l'entrainement Llama-Factory

In [ ]:
# Cellule 19 : Generation de dataset_info.json

# Lire le dataset_info.json existant de Llama-Factory et ajouter nos datasets
dataset_info_path = DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with open(dataset_info_path, "r") as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

# Enregistrement des datasets Pokemon
dataset_info["pokemon_stage1"] = {
    "file_name": "pokemon_stage1.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

dataset_info["pokemon_stage2"] = {
    "file_name": "pokemon_stage2.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

print("dataset_info.json mis a jour avec pokemon_stage1 et pokemon_stage2")

In [ ]:
# Cellule 20 : Configuration YAML Stage 1

stage1_yaml = """### Stage 1 : Entrainement LoRA sur donnees basse temperature

### Model
model_name_or_path: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
trust_remote_code: true
quantization_bit: 4

### Method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_target: all

### Dataset
dataset: pokemon_stage1
template: qwen3_nothink
cutoff_len: 2048
preprocessing_num_workers: 4

### Training
output_dir: saves/pokemon-stage1
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 5
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
gradient_checkpointing: true
logging_steps: 5
save_steps: 50
save_total_limit: 2
"""

with open(WORK_DIR / "stage1_train.yaml", "w") as f:
    f.write(stage1_yaml)

print("stage1_train.yaml cree")
print(stage1_yaml)

In [ ]:
# Cellule 21 : Configuration YAML Stage 2

stage2_yaml = """### Stage 2 : Entrainement LoRA sur donnees haute temperature
### Charge l'adapter LoRA du Stage 1

### Model
model_name_or_path: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit
adapter_name_or_path: saves/pokemon-stage1
trust_remote_code: true
quantization_bit: 4

### Method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_target: all

### Dataset
dataset: pokemon_stage2
template: qwen3_nothink
cutoff_len: 2048
preprocessing_num_workers: 4

### Training
output_dir: saves/pokemon-stage2
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 3
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
gradient_checkpointing: true
logging_steps: 5
save_steps: 50
save_total_limit: 2
"""

with open(WORK_DIR / "stage2_train.yaml", "w") as f:
    f.write(stage2_yaml)

print("stage2_train.yaml cree")
print(stage2_yaml)

In [ ]:
# Cellule 22 : Verification des fichiers de configuration

print("Verification des fichiers :")
for fname in ["data/pokemon_stage1.json", "data/pokemon_stage2.json",
              "data/dataset_info.json", "stage1_train.yaml", "stage2_train.yaml"]:
    fpath = WORK_DIR / fname
    if fpath.exists():
        size = fpath.stat().st_size
        print(f"  OK  {fname} ({size} bytes)")
    else:
        print(f"  MANQUANT  {fname}")

---
## Phase 6 : Entrainement

In [ ]:
# Cellule 23 : Liberation de la memoire GPU avant entrainement

print("Liberation de la memoire GPU...")
print(f"Avant : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

del student_model
del student_tokenizer
gc.collect()
torch.cuda.empty_cache()

print(f"Apres : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Memoire GPU liberee.")

In [ ]:
# Cellule 24 : Entrainement Stage 1

print("Lancement de l'entrainement Stage 1...")
!cd /content/LLaMA-Factory && llamafactory-cli train stage1_train.yaml

In [ ]:
# Cellule 25 : Entrainement Stage 2

print("Lancement de l'entrainement Stage 2...")
!cd /content/LLaMA-Factory && llamafactory-cli train stage2_train.yaml

---
## Phase 7 : Analyse des resultats d'entrainement

In [ ]:
# Cellule 26 : Courbes de loss

import json

def parse_trainer_logs(log_path):
    """Parse trainer_log.jsonl pour extraire les courbes de loss."""
    steps = []
    losses = []
    with open(log_path, "r") as f:
        for line in f:
            entry = json.loads(line.strip())
            if "loss" in entry and "current_steps" in entry:
                steps.append(entry["current_steps"])
                losses.append(entry["loss"])
    return steps, losses

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stage 1
log1_path = WORK_DIR / "saves" / "pokemon-stage1" / "trainer_log.jsonl"
if log1_path.exists():
    steps1, losses1 = parse_trainer_logs(log1_path)
    axes[0].plot(steps1, losses1, color="steelblue", linewidth=1.5)
    axes[0].set_title("Stage 1 - Loss (tau=0.3)")
    axes[0].set_xlabel("Steps")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "Log Stage 1 non trouve", ha='center', va='center')

# Stage 2
log2_path = WORK_DIR / "saves" / "pokemon-stage2" / "trainer_log.jsonl"
if log2_path.exists():
    steps2, losses2 = parse_trainer_logs(log2_path)
    axes[1].plot(steps2, losses2, color="coral", linewidth=1.5)
    axes[1].set_title("Stage 2 - Loss (tau=0.9)")
    axes[1].set_xlabel("Steps")
    axes[1].set_ylabel("Loss")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Log Stage 2 non trouve", ha='center', va='center')

plt.tight_layout()
plt.savefig(WORK_DIR / "loss_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("Courbes de loss sauvegardees dans loss_curves.png")

---
## Phase 8 : Test du modele distille

In [ ]:
# Cellule 27 : Chargement du modele distille (base + LoRA Stage 2)

from peft import PeftModel

print("Chargement du modele de base...")
base_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Chargement de l'adapter LoRA Stage 2...")
adapter_path = WORK_DIR / "saves" / "pokemon-stage2"
distilled_model = PeftModel.from_pretrained(base_model, str(adapter_path))
distilled_model.eval()

print(f"Modele distille charge.")
print(f"Memoire GPU : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cellule 28 : Comparaison qualitative - base vs distille

TEST_QUESTIONS = [
    "Pourquoi Carchacrok est-il si populaire en competitif Pokemon ? Analyse ses stats, son typing et ses capacites cles.",
    "Un Pokemon de type Eau/Sol comme Laggron a-t-il des faiblesses ? Si oui, lesquelles et comment les exploiter ?",
    "Construis un core defensif efficace autour de Noctali. Quels Pokemon le complementent et pourquoi ?"
]

def generate_response(model, tokenizer, question, max_new_tokens=512):
    """Genere une reponse avec le modele."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Generation des reponses pour chaque modele
print("Comparaison qualitative : Modele de base vs Modele distille\n")

comparison_results = []

for i, question in enumerate(TEST_QUESTIONS):
    print(f"{'='*80}")
    print(f"Question {i+1} : {question}")
    print(f"{'='*80}")

    # Reponse du modele de base (desactiver l'adapter LoRA)
    distilled_model.disable_adapter_layers()
    base_response = generate_response(distilled_model, base_tokenizer, question)

    # Reponse du modele distille (reactiver l'adapter LoRA)
    distilled_model.enable_adapter_layers()
    distilled_response = generate_response(distilled_model, base_tokenizer, question)

    comparison_results.append({
        "question": question,
        "base": base_response,
        "distilled": distilled_response
    })

    print(f"\n--- Modele de base ---")
    print(base_response[:600])
    if len(base_response) > 600:
        print(f"... ({len(base_response) - 600} chars de plus)")

    print(f"\n--- Modele distille ---")
    print(distilled_response[:600])
    if len(distilled_response) > 600:
        print(f"... ({len(distilled_response) - 600} chars de plus)")
    print()

In [ ]:
# Cellule 29 : Evaluation quantitative

def score_response(response):
    """
    Scoring automatique d'une reponse Pokemon.
    Criteres :
    - Presence de <reasoning>...</reasoning>
    - Longueur de la reponse
    - Presence de mots-cles Pokemon
    - Etapes numerotees dans le raisonnement
    """
    score = 0
    details = {}

    # 1. Presence de raisonnement structure (0-3 points)
    has_reasoning = "<reasoning>" in response and "</reasoning>" in response
    details["reasoning_tags"] = has_reasoning
    if has_reasoning:
        score += 3
    elif any(tag in response.lower() for tag in ["etape", "step", "raisonnement", "analyse"]):
        score += 1
        details["reasoning_tags"] = "partial"

    # 2. Longueur adequate (0-2 points)
    length = len(response)
    details["length"] = length
    if length >= 500:
        score += 2
    elif length >= 200:
        score += 1

    # 3. Mots-cles Pokemon (0-3 points)
    pokemon_keywords = [
        "type", "stab", "faiblesse", "resistance", "immunite",
        "attaque", "defense", "vitesse", "pv", "capacite",
        "evolution", "stat", "bst", "efficace", "super efficace",
        "competitif", "equipe", "synergie", "couverture",
        "sweeper", "wall", "pivot", "degats", "puissance"
    ]
    found_keywords = [kw for kw in pokemon_keywords if kw in response.lower()]
    details["keywords_found"] = len(found_keywords)
    if len(found_keywords) >= 8:
        score += 3
    elif len(found_keywords) >= 4:
        score += 2
    elif len(found_keywords) >= 1:
        score += 1

    # 4. Etapes numerotees (0-2 points)
    numbered_steps = len(re.findall(r'\d+[.\)]\s', response))
    details["numbered_steps"] = numbered_steps
    if numbered_steps >= 3:
        score += 2
    elif numbered_steps >= 1:
        score += 1

    details["total_score"] = score
    return score, details

# Evaluation de toutes les reponses
print("Evaluation quantitative\n")
print(f"{'Question':<12} {'Base':>8} {'Distille':>10} {'Delta':>8}")
print("-" * 42)

total_base = 0
total_distilled = 0

eval_details = []

for i, comp in enumerate(comparison_results):
    score_base, details_base = score_response(comp["base"])
    score_dist, details_dist = score_response(comp["distilled"])

    delta = score_dist - score_base
    delta_str = f"+{delta}" if delta > 0 else str(delta)

    print(f"Question {i+1:<4} {score_base:>6}/10 {score_dist:>8}/10 {delta_str:>8}")

    total_base += score_base
    total_distilled += score_dist

    eval_details.append({
        "question": comp["question"][:50],
        "base_score": score_base,
        "distilled_score": score_dist,
        "base_details": details_base,
        "distilled_details": details_dist
    })

n = len(comparison_results)
print("-" * 42)
avg_delta = (total_distilled - total_base) / n
avg_delta_str = f"+{avg_delta:.1f}" if avg_delta > 0 else f"{avg_delta:.1f}"
print(f"{'Moyenne':<12} {total_base/n:>6.1f}/10 {total_distilled/n:>8.1f}/10 {avg_delta_str:>8}")

In [ ]:
# Cellule 30 : Tableau recapitulatif des metriques

print("\n" + "=" * 70)
print("TABLEAU RECAPITULATIF")
print("=" * 70)

print(f"\n{'Metrique':<35} {'Base':>12} {'Distille':>12}")
print("-" * 62)

# Calculer les metriques moyennes
base_lengths = [len(c['base']) for c in comparison_results]
dist_lengths = [len(c['distilled']) for c in comparison_results]
base_reasoning = sum(1 for c in comparison_results if '<reasoning>' in c['base'])
dist_reasoning = sum(1 for c in comparison_results if '<reasoning>' in c['distilled'])
base_steps = [len(re.findall(r'\d+[.\)]\s', c['base'])) for c in comparison_results]
dist_steps = [len(re.findall(r'\d+[.\)]\s', c['distilled'])) for c in comparison_results]

print(f"{'Score moyen (/10)':<35} {total_base/n:>10.1f} {total_distilled/n:>12.1f}")
print(f"{'Longueur moyenne (chars)':<35} {np.mean(base_lengths):>10.0f} {np.mean(dist_lengths):>12.0f}")
print(f"{'<reasoning> present':<35} {base_reasoning:>10}/{n} {dist_reasoning:>10}/{n}")
print(f"{'Etapes numerotees (moy)':<35} {np.mean(base_steps):>10.1f} {np.mean(dist_steps):>12.1f}")

# Details par critere
print(f"\n{'Critere':<35} {'Base moy':>12} {'Dist. moy':>12}")
print("-" * 62)
for critere in ["reasoning_tags", "length", "keywords_found", "numbered_steps"]:
    base_vals = [d["base_details"][critere] for d in eval_details]
    dist_vals = [d["distilled_details"][critere] for d in eval_details]
    if isinstance(base_vals[0], bool) or isinstance(base_vals[0], str):
        base_str = f"{sum(1 for v in base_vals if v is True)}/{n}"
        dist_str = f"{sum(1 for v in dist_vals if v is True)}/{n}"
    else:
        base_str = f"{np.mean([float(v) for v in base_vals]):.1f}"
        dist_str = f"{np.mean([float(v) for v in dist_vals]):.1f}"
    print(f"{critere:<35} {base_str:>12} {dist_str:>12}")

# Info dataset
print(f"\n{'='*70}")
print("DONNEES D'ENTRAINEMENT")
print(f"{'='*70}")
print(f"{'Stage 1 (tau=0.3) exemples':<35} {len(pokemon_stage1):>12}")
print(f"{'Stage 2 (tau=0.9) exemples':<35} {len(pokemon_stage2):>12}")
print(f"{'Total questions Pokemon':<35} {len(all_questions):>12}")

---
## Phase 10 : Conclusion

### Resultats

Ce notebook a implemente le pipeline DASD complet pour distiller les capacites de raisonnement Pokemon d'un modele enseignant vers Qwen3-4B :

1. **Generation de donnees** : 28 questions Pokemon en 5 categories, generees a deux temperatures (0.3 et 0.9) via l'API Infomaniak
2. **DAS** : Le Divergence-Aware Sampling a permis de filtrer les exemples ou le modele etudiant a le plus a apprendre
3. **Entrainement LoRA** : Deux stages d'entrainement avec Llama-Factory, le second chargeant l'adapter du premier
4. **Evaluation** : Comparaison base vs distille montrant l'apport de la distillation

### Limites

- **Taille du dataset** : 28 questions est un minimum pour un TP ; un dataset plus large (100+) ameliorerait significativement les resultats
- **Qualite du teacher** : La qualite des reponses depend fortement du modele enseignant disponible sur l'API Infomaniak
- **Alignement des tokens** : L'alignement phrase-par-phrase entre les tokenizers teacher et student est approximatif (curseur caractere)
- **Evaluation** : Le scoring automatique est un proxy ; une evaluation humaine ou via un LLM evaluateur serait plus fiable
- **Overfitting** : Avec un petit dataset, le risque d'overfitting est reel malgre l'utilisation de LoRA

### Ameliorations possibles

- Augmenter le nombre de questions et les diversifier
- Ajouter des generations multiples par question pour plus de diversite
- Implementer un alignement token-par-token plus precis
- Utiliser un LLM evaluateur (GPT-4) pour la notation qualitative
- Experimenter avec d'autres valeurs de LoRA rank (16, 32)